# Data Quality & Reconciliation

Accounts for every record from raw export to what lands on the map, so you can
answer *"why isn't property X on the map?"*. The funnel:

```
raw merged rows
  - unparseable Sale Date        (dropped)
  - Sale Price < $1,000          (non-arms-length: $0-$10 transfers, dropped)
  = clean (used for trend analysis)
  - address not geocoded         (no lat/lon, can't map)
  = mappable (all years/classes)
  - outside the active FILTERS   (year / class / beds)
  = shown on the map
```

Most "missing from map" cases are the **last** step (a filter), not bad data —
e.g. *77 Sycamore Ave* is geocoded fine but its only real sale was 2013, so the
default `year_min=2020` map hides it.

In [ ]:
import re
import pandas as pd
from pathlib import Path

ROOT = Path('..')
df = pd.read_csv(ROOT / 'data' / 'merged.csv', dtype=str)
cache = pd.read_csv(ROOT / 'output' / 'geocode_cache.csv', dtype={'address': str})

df['sale_date'] = pd.to_datetime(df['Sale Date'], format='%m/%d/%Y', errors='coerce')
df['price'] = pd.to_numeric(df['Sale Price'], errors='coerce')
df['year'] = df['sale_date'].dt.year
df['addr'] = df['Property Address'].str.strip()

matched = set(cache.loc[cache['matched'], 'address'])

# per-row drop reasons (a row can fail more than one; each is reported).
# A non-numeric/missing price counts as non-arms-length too: NaN < 1000 is False,
# so without this such rows would slip through here but get dropped silently by
# the trend/map notebooks (median & plotting ignore NaN) -> count them honestly.
df['bad_date']     = df['sale_date'].isna()
df['non_arms']     = df['price'].isna() | (df['price'] < 1000)
df['not_geocoded'] = ~df['addr'].isin(matched)
df['clean']    = ~df['bad_date'] & ~df['non_arms']
df['mappable'] = df['clean'] & ~df['not_geocoded']
print(f'loaded {len(df):,} raw rows; {len(matched):,} addresses geocoded')

## 1. The funnel — record counts at each stage

In [ ]:
n = len(df)
clean = df[df['clean']]
mappable = df[df['mappable']]
funnel = pd.DataFrame([
    ('raw merged rows', n),
    ('  - unparseable Sale Date', -df['bad_date'].sum()),
    ('  - price missing/non-numeric', -df['price'].isna().sum()),
    ('  - Sale Price < $1,000 (non-arms-length)', -(df['price'] < 1000).sum()),
    ('= clean (trend analysis)', len(clean)),
    ('  - address not geocoded', -(clean['not_geocoded'].sum())),
    ('= mappable (all years/classes)', len(mappable)),
], columns=['stage', 'rows']).set_index('stage')
funnel

## 2. What gets dropped, and why

Note the funnel reasons overlap (a $1 sale may also be ungeocodable); these
tallies count each reason independently against the raw rows.

In [ ]:
drops = pd.DataFrame({
    'rows': [df['bad_date'].sum(), df['non_arms'].sum(),
             df['not_geocoded'].sum(), df.loc[df['clean'], 'not_geocoded'].sum()],
}, index=['unparseable date', 'price < $1,000',
          'not geocoded (any row)', 'not geocoded (among clean)'])
drops['% of raw'] = (100 * drops['rows'] / len(df)).round(1)
drops

## 3. Addresses that fail to geocode

Mostly records with no house number (commercial / condo / street-only).

In [ ]:
ungeo = df.loc[df['clean'] & df['not_geocoded'], 'addr']
no_number = ungeo[~ungeo.str.match(r'^\d')]
print(f"clean rows that don't geocode: {len(ungeo):,} ({ungeo.nunique():,} unique addresses)")
print(f"  of those, addresses with no leading house number: {no_number.nunique():,}")
ungeo.value_counts().head(20)

## 4. "Why isn't this address on the map?"

Pass an address (substring, case-insensitive) and the **active map filters** to
see, per sale, exactly which stage includes or excludes it.

In [ ]:
def diagnose(query, property_class='Residential', year_min=2020, year_max=None, beds=None):
    """Explain, per sale of `query`, whether it reaches the map under these filters."""
    m = df[df['addr'].str.contains(query, case=False, na=False)].copy()
    if m.empty:
        return f"No rows match '{query}' in the raw data at all."

    def reason(r):
        if r['bad_date']:  return 'DROPPED: unparseable sale date'
        if r['non_arms']:  return 'DROPPED: price < $1,000 (non-arms-length)'
        if r['not_geocoded']: return 'DROPPED: address not geocoded'
        if property_class is not None and r['Property Class'] != property_class:
            return f"hidden by filter: class != {property_class}"
        if year_min is not None and r['year'] < year_min:
            return f'hidden by filter: year {int(r["year"])} < {year_min}'
        if year_max is not None and r['year'] > year_max:
            return f'hidden by filter: year {int(r["year"])} > {year_max}'
        if beds is not None and pd.to_numeric(r['Beds'], errors='coerce') != beds:
            return f'hidden by filter: beds != {beds}'
        return 'ON MAP'

    m['verdict'] = m.apply(reason, axis=1)
    return m.sort_values('sale_date', ascending=False)[
        ['addr', 'Sale Date', 'Sale Price', 'Property Class', 'year', 'verdict']]

diagnose('77 Sycamore')

Same address, but widen the year filter — now it's on the map:

In [ ]:
diagnose('77 Sycamore', year_min=None)